# Grounding en LTN (suite)

Ce tutoriel explique comment grounder les connecteurs et les quantificateurs en LTN. Il suppose une certaine familiarité avec le premier tutoriel sur le grounding des symboles non-logiques (constantes, variables, fonctions et prédicats).

In [1]:
import ltn
import numpy as np
import torch

## Connecteurs

On a vu en théorie que les connecteurs logiques classiques ($\lnot,\land,\lor,\implies$) ne peuvent pas être utilisés tels quels en LTN, puisque leurs tables de vérité ne sont définies que pour des valeurs strictement dans $\{0,1\}$. LTN les remplace donc par des opérateurs de **logique floue**, définis sur tout l'intervalle continu $[0,1]$.

Il existe plusieurs familles d'opérateurs flous valides (Gödel, Produit, Łukasiewicz), mais elles ne sont pas équivalentes du point de vue de l'entraînement. Certaines, comme le $\min$/$\max$ de Gödel, ne font circuler le gradient que sur un seul argument à la fois, ce qui peut ralentir ou bloquer l'apprentissage. C'est pourquoi LTN recommande en général la configuration suivante, où $u$ et $v$ désignent deux degrés de vérité dans $[0,1]$ :

* la négation standard : $\lnot u = 1-u$ ;
* la t-norme produit pour la conjonction : $u \land v = uv$ ;
* la t-conorme produit (somme probabiliste) pour la disjonction : $u \lor v = u+v-uv$ ;
* l'implication de Reichenbach : $u \implies v = 1-u+uv$.

Ce choix n'est pas arbitraire. Contrairement au $\min$/$\max$, le produit a un gradient qui se répartit sur les deux arguments à la fois, ce qui en fait un choix numériquement plus stable pour l'entraînement. Pour approfondir ce compromis entre fidélité logique et comportement du gradient, voir le notebook complémentaire `2b-operators-and-gradients.ipynb`.

Créer un connecteur en LTN est très simple : le constructeur `Connective()` prend en entrée une sémantique floue, unaire ou binaire, choisie dans le module `ltn.fuzzy_ops`. Dans ce tutoriel, on utilise exactement la configuration recommandée ci-dessus.

In [2]:
Not = ltn.Connective(ltn.fuzzy_ops.NotStandard())
And = ltn.Connective(ltn.fuzzy_ops.AndProd())
Or = ltn.Connective(ltn.fuzzy_ops.OrProbSum())
Implies = ltn.Connective(ltn.fuzzy_ops.ImpliesReichenbach())

Le wrapper `ltn.Connective` ne se contente pas d'appliquer la formule floue brute, il gère aussi la combinaison de sous-formules qui ne portent pas sur les mêmes variables. Or, comme on l'a vu dans le premier notebook, deux sous-formules impliquant des variables différentes peuvent avoir des tenseurs de formes différentes (une formule sur $x$ seul aura une forme $(n_x,)$, une formule sur $x$ et $y$ aura une forme $(n_x, n_y)$). Le connecteur doit alors étendre automatiquement ces formes pour les rendre compatibles avant d'appliquer l'opérateur terme à terme. C'est ce qu'on appelle le **broadcasting**, qu'on va observer concrètement dans les exemples qui suivent.

Pour illustrer ce mécanisme, on définit deux variables comportant un nombre différent d'individus, deux constantes, ainsi qu'un prédicat mesurant la similarité entre deux points de $\mathbb{R}^2$, la même construction $\exp(-\|x-y\|)$ que dans le premier notebook, qui vaut $1$ quand les deux points coïncident et tend vers $0$ à mesure qu'ils s'éloignent.

In [3]:
x = ltn.Variable('x', torch.randn((10, 2))) # 10 values in R²
y = ltn.Variable('y', torch.randn((5, 2))) # 5 values in R²

c1 = ltn.Constant(torch.tensor([0.5, 0.0]))
c2 = ltn.Constant(torch.tensor([4.0, 2.0]))

Eq = ltn.Predicate(func=lambda x, y: torch.exp(-torch.norm(x - y, dim=1))) # predicate measuring similarity

Eq(c1, c2).value

tensor(0.0178)

Vérifions maintenant le comportement des connecteurs logiques sur des cas concrets, en gardant à l'esprit les formules rappelées plus haut.

Les deux derniers exemples sont particulièrement instructifs, car ils mettent en évidence deux situations différentes vis-à-vis des variables en jeu.

Dans `And(Eq(x, c1), Eq(x, c2))`, seule la variable $x$ apparaît dans les deux sous-formules (chaque argument de `Eq` étant $x$ associé à une constante), donc les deux résultats ont déjà la même forme $(10,)$. Il n'y a pas besoin de broadcasting : `And` les combine directement terme à terme, individu par individu.

Dans `Or(Eq(x, c1), Eq(x, y))`, la situation est différente. Le premier argument, `Eq(x,c1)`, ne dépend que de $x$ et a donc la forme $(10,)$, tandis que le second, `Eq(x,y)`, dépend à la fois de $x$ et de $y$ et a la forme $(10,5)$, rappel du premier notebook : une évaluation sur deux variables produit toutes les combinaisons possibles, pas un simple alignement. Le connecteur détecte cet écart de forme et étend automatiquement le premier résultat le long de l'axe manquant (celui de $y$), en répétant chaque valeur pour chacun des 5 individus de $y$, avant d'appliquer `Or` terme à terme. On obtient ainsi une forme finale de $(10,5)$.

In [4]:
Not(Eq(c1, c2)).value

tensor(0.9822)

In [5]:
Implies(Eq(c1, c2), Eq(c2, c1)).value

tensor(0.9825)

In [6]:
# Notice the dimension of the outcome: the result is evaluated for every x.
And(Eq(x, c1), Eq(x, c2)).shape()

torch.Size([10])

In [7]:
# Notice the dimensions of the outcome: the result is evaluated for every x and y.
# Notice also that y did not appear in the 1st argument of `Or`;
# the connective broadcasts the results of its two arguments to match.
Or(Eq(x, c1), Eq(x, y)).shape()

torch.Size([10, 5])

Comme pour les prédicats et les fonctions, on accède au résultat de l'évaluation d'un connecteur via son attribut `value`, ou on consulte sa forme via `shape()`. Les connecteurs LTN renvoient eux aussi des instances de `LTNObject`.

## Quantificateurs

LTN prend en charge la quantification universelle et existentielle. Comme pour les connecteurs, ces quantificateurs ne peuvent pas être utilisés dans leur forme classique, puisque $\forall$ et $\exists$ supposent de vérifier une propriété sur un domaine entier, alors qu'en LTN une variable ne représente qu'un batch fini d'individus, avec des degrés de vérité continus dans $[0,1]$. LTN les remplace donc par des **opérateurs d'agrégation**, qui condensent les degrés de vérité d'un batch en une seule valeur.

Nous recommandons les deux opérateurs suivants, où $u_1,\dots,u_n$ désigne une liste de degrés de vérité dans $[0,1]$ :

* pour la quantification existentielle ($\exists$), la moyenne généralisée (`pMean`) :
$$\mathrm{pM}(u_1,\dots,u_n) = \left(\frac{1}{n}\sum_{i=1}^n u_i^p\right)^{1/p}, \qquad p\geq 1$$

* pour la quantification universelle ($\forall$), la moyenne généralisée des écarts à la vérité (`pMeanError`) :
$$\mathrm{pME}(u_1,\dots,u_n) = 1 - \left(\frac{1}{n}\sum_{i=1}^n (1-u_i)^p\right)^{1/p}, \qquad p\geq 1$$

Ces deux formules ne sont pas choisies au hasard : `pMean` approxime le maximum et `pMeanError` approxime le minimum, ce qui correspond bien à l'esprit logique de $\exists$ (il suffit qu'un seul individu satisfasse la propriété) et de $\forall$ (c'est le pire cas qui doit être vérifié). On démontrera plus loin, dans une section dédiée, pourquoi ces formules convergent effectivement vers le maximum et le minimum.

Créer un quantificateur en LTN est très simple : le constructeur `Quantifier()` prend en entrée une sémantique d'agrégation, ainsi qu'un caractère indiquant le type de quantification associé (`"e"` pour l'existentielle, `"f"` pour l'universelle). Dans cet exemple, on crée les deux quantificateurs avec les sémantiques recommandées ci-dessus.

In [8]:
Forall = ltn.Quantifier(ltn.fuzzy_ops.AggregPMeanError(p=2), quantifier="f")
Exists = ltn.Quantifier(ltn.fuzzy_ops.AggregPMean(p=2), quantifier="e")

Le wrapper `ltn.Quantifier` permet d'utiliser ces agrégateurs directement sur des formules LTN. Il se charge de sélectionner, dans le tenseur représentant la formule, les dimensions à agréger en fonction des variables passées en argument : quantifier sur une variable revient concrètement à appliquer l'agrégateur le long de l'axe correspondant, ce qui fait disparaître cet axe du résultat.

Dans cet exemple, on reprend des variables et un prédicat similaires à ceux de l'exemple précédent sur les connecteurs.

In [9]:
x = ltn.Variable('x', torch.randn((10, 2))) # 10 values in R²
y = ltn.Variable('y', torch.randn((5, 2))) # 5 values in R²

Eq = ltn.Predicate(func=lambda x, y: torch.exp(-torch.norm(x - y, dim=1))) # predicate measuring similarity

Eq(x, y).shape()

torch.Size([10, 5])

Appliquons maintenant quelques quantificateurs à la formule, et observons l'effet sur la sortie et sur sa forme.

Dans le premier cas, la quantification porte sur $x$ seul : l'axe correspondant à $x$ disparaît, et il ne reste que l'axe de $y$. La forme obtenue est donc $(5,)$, puisque $y$ compte 5 individus, et le résultat reste une fonction de $y$ (il indique, pour chaque $y$, à quel degré la propriété est vraie pour tous les $x$).

Dans les trois cas suivants, la quantification porte sur les deux variables à la fois : les deux axes disparaissent, et le résultat est un scalaire unique, un seul degré de satisfaction global pour toute la formule.

In [10]:
Forall(x, Eq(x, y)).shape()

torch.Size([5])

In [11]:
Forall([x, y], Eq(x, y)).value

tensor(0.1521)

In [12]:
Exists([x, y], Eq(x, y)).value

tensor(0.2211)

In [13]:
Forall(x, Exists(y, Eq(x, y))).value

tensor(0.1846)

Comme pour les prédicats, les fonctions et les connecteurs, on accède au résultat via l'attribut `value`, ou à sa forme via `shape()`. Les quantificateurs LTN renvoient eux aussi des instances de `LTNObject`.

Un point de syntaxe à retenir : quand la quantification porte sur une seule variable, on peut la donner directement au quantificateur, telle quelle. En revanche, dès que la quantification porte sur plusieurs variables, il faut les regrouper dans une liste, comme dans les deuxième et troisième exemples ci-dessus.

### Démonstration : pourquoi `pMean` tend vers le maximum, et `pMeanError` vers le minimum

On avait annoncé plus haut que ces deux formules ne sont pas choisies arbitrairement : `pMean` approxime le maximum et `pMeanError` approxime le minimum. Démontrons-le proprement.

#### Partie 1 : `pMean` tend vers le maximum

On cherche à montrer que, pour $u_1,\dots,u_n \in [0,1]$ :

$$\lim_{p\to+\infty} \left(\frac{1}{n}\sum_{i=1}^n u_i^p\right)^{1/p} = \max(u_1,\dots,u_n)$$

Notons $M = \max(u_1,\dots,u_n)$. On exclut le cas trivial $M=0$ (où tous les $u_i$ sont nuls, et où `pMean` vaut $0$), et on suppose donc $M>0$.

Puisque $M\neq 0$, on peut factoriser :

$$\sum_{i=1}^n u_i^p = M^p \sum_{i=1}^n \left(\frac{u_i}{M}\right)^p$$

d'où :

$$\left(\frac{1}{n}\sum_{i=1}^n u_i^p\right)^{1/p} = M \cdot \left(\frac{1}{n}\sum_{i=1}^n \left(\frac{u_i}{M}\right)^p\right)^{1/p}$$

Posons $r_i = u_i/M \in [0,1]$ (puisque $u_i \leq M$ par définition de $M$). Deux cas se présentent :

- pour l'indice $i^*$ qui réalise le maximum, $r_{i^*} = 1$, donc $r_{i^*}^p = 1$ pour tout $p$ ;
- pour tout autre indice, $r_i \in [0,1)$, et une fraction strictement inférieure à 1 élevée à une puissance qui grandit indéfiniment tend vers 0 : $\lim_{p\to\infty} r_i^p = 0$.

En notant $k\geq 1$ le nombre d'indices réalisant le maximum (en général $k=1$, mais des égalités restent possibles) :

$$\lim_{p\to\infty} \sum_{i=1}^n r_i^p = \underbrace{1+\dots+1}_{k\text{ fois}} + \underbrace{0+\dots+0}_{n-k\text{ fois}} = k \qquad\Longrightarrow\qquad \lim_{p\to\infty} \frac{1}{n}\sum_{i=1}^n r_i^p = \frac{k}{n}$$

Reste à passer à la racine $p$-ième de cette limite. Comme $k/n$ est une constante strictement positive indépendante de $p$, on utilise le résultat classique $\lim_{p\to\infty} c^{1/p} = 1$ pour tout $c>0$ (en écrivant $c^{1/p}=e^{\frac{1}{p}\ln c}$, et en remarquant que $\frac{1}{p}\ln c \to 0$). D'où :

$$\lim_{p\to\infty} \left(\frac{1}{n}\sum_{i=1}^n r_i^p\right)^{1/p} = 1$$

et finalement :

$$\lim_{p\to\infty} \left(\frac{1}{n}\sum_{i=1}^n u_i^p\right)^{1/p} = M\cdot 1 = \max(u_1,\dots,u_n)$$

#### Partie 2 : `pMeanError` tend vers le minimum

On veut montrer que :

$$\lim_{p\to+\infty} \left[1 - \left(\frac{1}{n}\sum_{i=1}^n (1-u_i)^p\right)^{1/p}\right] = \min(u_1,\dots,u_n)$$

L'astuce consiste à appliquer directement le résultat de la partie 1, non pas à la suite $(u_i)$, mais à la suite des compléments $(1-u_i)$, qui appartiennent eux aussi à $[0,1]$ :

$$\lim_{p\to\infty} \left(\frac{1}{n}\sum_{i=1}^n (1-u_i)^p\right)^{1/p} = \max_i(1-u_i)$$

Or $\max_i(1-u_i) = 1-\min_i(u_i)$ :

<details>
<summary><b>Justification mathématique</b></summary>

**Proposition.** Soit $(u_i)_i$ une famille finie de réels, et soit $m=\min_i(u_i)$. Alors $\max_i(1-u_i)=1-m$.

**Démonstration.** Par définition de $m$, on a $u_i\geq m$ pour tout $i$. En multipliant par $-1$ puis en ajoutant $1$ de part et d'autre, il vient $1-u_i \leq 1-m$ pour tout $i$. Le réel $1-m$ est donc un majorant de la famille $(1-u_i)_i$, si bien que $\max_i(1-u_i)\leq 1-m$.

Le minimum d'une famille finie étant toujours atteint, il existe un indice $i^*$ tel que $u_{i^*}=m$, d'où $1-u_{i^*}=1-m$. Ce réel figure donc parmi les termes de la famille $(1-u_i)_i$, et le maximum d'une famille est toujours supérieur ou égal à chacun de ses termes, donc $\max_i(1-u_i)\geq 1-m$.

Les deux inégalités entraînent l'égalité annoncée : $\max_i(1-u_i)=1-m=1-\min_i(u_i)$.
</details>

D'où :

$$\lim_{p\to\infty} \left(\frac{1}{n}\sum_{i=1}^n (1-u_i)^p\right)^{1/p} = 1-\min_i(u_i)$$

$$\Rightarrow 1 - \lim_{p\to\infty} \left(\frac1n\sum_i(1-u_i)^p\right)^{1/p} = \min_i(u_i)$$

Et par conséquent :

$$\lim_{p\to\infty}\left[1 - \left(\frac{1}{n}\sum_{i=1}^n (1-u_i)^p\right)^{1/p}\right] = \min_i(u_i)$$

#### Ce qu'il faut retenir

Élever à une puissance $p$ de plus en plus grande amplifie l'écart relatif entre le plus grand terme et les autres, jusqu'à ce que seul ce terme dominant subsiste dans la somme. C'est ce mécanisme purement algébrique qui explique la convergence de `pMean` vers le maximum, et, via le passage aux compléments, la convergence de `pMeanError` vers le minimum. Pour un $p$ fini, ces formules restent cependant des versions lissées et différentiables, qui préservent un gradient sur tous les termes, contrairement au $\max$/$\min$ strict dont le gradient ne circule que sur l'élément gagnant.

## Sémantique des quantificateurs

La sémantique `pMean` peut être vue comme un maximum adouci, dont le comportement dépend de l'hyperparamètre $p$ :
* pour $p = 1$, l'opérateur est une simple moyenne ;
* quand $p \to +\infty$, l'opérateur se rapproche du maximum strict.

De la même façon, `pMeanError` peut être vue comme un minimum adouci :
* pour $p = 1$, l'opérateur est une simple moyenne ;
* quand $p \to +\infty$, l'opérateur se rapproche du minimum strict.

Le paramètre $p$ règle en réalité un compromis entre fidélité logique et tolérance aux cas difficiles du batch.

Quand $p$ est élevé, `pMeanError` se rapproche du vrai minimum, et le quantificateur $\forall$ retrouve son sens logique strict : si un seul individu du batch a un faible degré de vérité, le résultat global chute fortement, exactement comme on l'attend d'un "pour tout" rigoureux. C'est un comportement fidèle, mais exigeant : pour obtenir un score global élevé, il faut que tous les individus, sans exception, aient un degré de vérité élevé. Symétriquement, `pMean` se rapproche du vrai maximum, et $\exists$ redevient facile à satisfaire : il suffit d'un seul bon individu pour que le résultat global soit élevé, peu importe la qualité des autres.

Quand $p$ se rapproche de 1, les deux opérateurs se rapprochent au contraire d'une simple moyenne, où chaque individu pèse à peu près également dans le résultat. C'est ici que les cas extrêmes cessent de dominer : un individu très mauvais peut être compensé par plusieurs bons individus, et inversement. Cette compensation rend $\forall$ artificiellement plus tolérant, un individu défaillant ne suffit plus à faire chuter tout le résultat, et $\exists$ artificiellement plus exigeant, un seul bon individu ne suffit plus à porter le résultat. Cette compensation est donc un écart volontaire par rapport à la sémantique logique stricte de $\forall$ et $\exists$, utile en pratique pour absorber le bruit dans les données ou pour obtenir un gradient mieux réparti à l'entraînement, mais elle s'éloigne d'autant plus de leur vrai sens logique que $p$ est proche de 1.

Le choix de $p$ dépend donc de l'objectif : un $p$ élevé convient à une formule qui doit rester logiquement stricte et ne tolérer aucune exception, tandis qu'un $p$ proche de 1 convient à une formule plus robuste face à des données bruitées ou à des valeurs aberrantes, au prix d'un écart voulu par rapport à la sémantique classique.

Des choix différents de $p$ peuvent avoir des conséquences importantes sur l'entraînement (voir `2b-operators-and-gradients.ipynb`).

On peut fixer une valeur par défaut pour $p$ à l'initialisation de l'opérateur, ou en spécifier une différente à chaque appel. Dans l'exemple suivant, on utilise les quantificateurs avec différentes valeurs de $p$.

In [14]:
Forall(x, Eq(x, c1), p=2).value

tensor(0.1942)

In [15]:
Forall(x, Eq(x, c1), p=10).value

tensor(0.1232)

In [16]:
Exists(x, Eq(x, c1), p=2).value

tensor(0.3085)

In [17]:
Exists(x, Eq(x, c1), p=10).value

tensor(0.6013)

## Quantification diagonale

Jusqu'ici, évaluer un prédicat sur deux variables produisait systématiquement toutes les combinaisons possibles entre leurs individus. Ce comportement par défaut n'est pourtant pas toujours celui qu'on souhaite. Imaginons un dataset supervisé, où chaque donnée $x_i$ est associée à son propre label $l_i$ : on ne veut évaluer un prédicat que sur les paires qui appartiennent réellement au même exemple, $(x_0,l_0)$, $(x_1,l_1)$, et ainsi de suite, jamais sur des combinaisons croisées comme $(x_0,l_3)$, qui n'auraient aucun sens.

C'est exactement ce que permet `ltn.diag` : au lieu de croiser toutes les combinaisons, il ne conserve que les paires en **correspondance un à un**, comme le ferait un `zip` en Python plutôt que deux boucles imbriquées.

Cette correspondance un à un impose une condition évidente : les variables concernées doivent avoir le même nombre d'individus, sinon il n'existe pas de façon naturelle de les faire correspondre terme à terme.

On illustre ce mécanisme avec l'exemple suivant :
* la variable $x$ représente 100 individus dans $\mathbb{R}^{2\times2}$ ;
* la variable $l$ représente 100 labels encodés en one-hot dans $\mathbb{N}^3$ (3 classes possibles) ;
* $l$ est construite en cohérence avec $x$, de sorte que chaque paire $(x_i,l_i)$ décrit un exemple correct et cohérent du dataset ;
* le classifieur $C(x,l)$ renvoie un degré de confiance dans $[0,1]$ quant au fait que l'échantillon $x$ corresponde bien au label $l$.

In [18]:
# The values are generated at random, for the sake of illustration.
# In a real scenario, they would come from a dataset.
samples = torch.randn((100, 2, 2)) # 100 R^{2x2} values
labels = torch.randint(0, 3, size=(100,)) # 100 labels (class 0/1/2) that correspond to each sample
onehot_labels = torch.nn.functional.one_hot(labels, num_classes=3)

x = ltn.Variable("x", samples)

l = ltn.Variable("l", onehot_labels)

class ModelC(torch.nn.Module):
    def __init__(self):
        super(ModelC, self).__init__()
        self.elu = torch.nn.ELU()
        self.softmax = torch.nn.Softmax(dim=1)
        self.dense1 = torch.nn.Linear(4, 5)
        self.dense2 = torch.nn.Linear(5, 3)

    def forward(self, x, l):
        x = torch.flatten(x, start_dim=1) # aplatit la matrice 2 x 2 en un vecteur de taille 4
        x = self.elu(self.dense1(x))
        x = self.softmax(self.dense2(x))  # sortie : un vecteur de taille 3 (probabilités pour chaque classe); ex : le réseau sort [0.1, 0.7, 0.2] pour 𝑥_0 pour un label y_0 = [0, 1, 0] 
        return torch.sum(x * l, dim=1)    # on sélectionne la probabilité de la classe correcte (l) ;               alors ici on calcule [0.1, 0.7, 0.2]⋅[0, 1, 0] = (0 x 0.1) + (0.7 x 1) + (0.2 x 0) = 0.7 

C = ltn.Predicate(ModelC())

Sans `ltn.diag`, `C(x,l)` évalue toutes les combinaisons possibles entre les 100 individus de $x$ et les 100 de $l$, soit une matrice $100\times100$. Le problème saute aux yeux dès qu'on réduit l'exemple à 4 individus : sur les 16 paires que produirait ce croisement complet, seules les 4 paires de la diagonale, $(x_0,l_0)$, $(x_1,l_1)$, $(x_2,l_2)$, $(x_3,l_3)$, correspondent à de vrais exemples du dataset. Toutes les autres cases évaluent des combinaisons artificielles, comme demander si l'image $x_0$ correspond au label de l'exemple 1, une question qu'on n'a jamais voulu poser.

`ltn.diag(x, l)` corrige précisément ce problème : au lieu des 100×100 combinaisons, seules les 100 paires correspondantes sont conservées, celles qui décrivent un exemple réel du dataset. La forme du résultat passe alors de $(100,100)$ à $(100,)$.

Le mécanisme technique derrière ce changement de comportement est le même que celui vu dans le premier notebook pour distinguer les axes d'un résultat : `ltn.diag` donne temporairement à $x$ et $l$ le **même label interne**, préfixé par `diag_`. Puisque LTN se base sur les labels pour savoir si deux variables doivent être croisées ou traitées comme un seul axe avancé en parallèle, partager le même label suffit à faire basculer du croisement complet vers la correspondance un à un.

`ltn.undiag` permet de revenir en arrière : il restaure les labels d'origine des variables, et donc le comportement normal de croisement complet, comme le montre le dernier affichage ci-dessous.

In [19]:
print(C(x, l).shape()) # Computes the 100x100 combinations
ltn.diag(x, l) # sets the diag behavior for x and l
print(C(x, l).shape())# Computes the 100 zipped combinations
print(x.free_vars)
print(l.free_vars)
ltn.undiag(x, l) # resets the normal behavior
print(C(x, l).shape()) # Computes the 100x100 combinations

torch.Size([100, 100])
torch.Size([100])
['diag_x_l']
['diag_x_l']
torch.Size([100, 100])


En pratique, `ltn.diag` est conçu pour être utilisé conjointement avec un quantificateur. Chaque quantificateur appelle automatiquement `ltn.undiag` une fois l'agrégation effectuée, de sorte que les variables retrouvent leur comportement normal en dehors de cette formule précise. C'est pourquoi l'usage recommandé consiste à appliquer `ltn.diag` juste avant un quantificateur, comme dans l'exemple suivant.

Ici, $\text{Forall}([x,l], C(x,l))$ n'agrège que sur les 100 paires correspondantes, exactement la question qu'on veut poser dans un problème de classification supervisée : à quel degré le modèle attribue-t-il, en moyenne, une bonne confiance à chaque exemple du dataset associé à son véritable label ? Une fois cette agrégation effectuée, `x` et `l` retrouvent automatiquement leurs labels d'origine, comme le confirment les affichages de `free_vars` avant et après l'appel au quantificateur.

In [20]:
x, l = ltn.diag(x, l)
print(x.free_vars)
print(l.free_vars)
print(Forall([x, l], C(x, l)).value) # Aggregates only on the 100 "zipped" pairs.
                                    # Automatically calls `ltn.undiag` so the behavior of x/l is unchanged outside of this formula.
print(x.free_vars)
print(l.free_vars)

['diag_x_l']
['diag_x_l']
tensor(0.3384, grad_fn=<RsubBackward1>)
['x']
['l']


## Quantificateurs gardés

On souhaite parfois quantifier non pas sur l'ensemble des individus d'une variable, mais uniquement sur ceux qui satisfont une condition supplémentaire. Cette condition n'est pas un degré de vérité continu dans $[0,1]$ comme le sont les prédicats LTN habituels : c'est un véritable masque **booléen**, strictement $0$ ou $1$, qui sert uniquement à sélectionner quels individus participent à l'agrégation.

Soit $x$ une variable d'un domaine, et $m$ une fonction de masquage renvoyant une valeur booléenne pour chaque élément du domaine. On peut alors écrire des quantifications de la forme :

* $(\forall x : m(x))\ \phi(x)$, qui signifie « tout $x$ satisfaisant $m(x)$ satisfait aussi $\phi(x)$ » ;
* $(\exists x : m(x))\ \phi(x)$, qui signifie « il existe un $x$ satisfaisant $m(x)$ qui satisfait aussi $\phi(x)$ ».

Le masque $m$ peut lui-même dépendre d'autres variables présentes dans la formule. Par exemple, $\exists y\ (\forall x : m(x,y))\ \phi(x,y)$ est également un énoncé valide, où la condition sur $x$ dépend de $y$.

Illustrons cela avec l'exemple suivant, qui affirme qu'il existe une distance euclidienne $d$ en dessous de laquelle toutes les paires de points $x,y$ doivent être considérées comme similaires :

$$\exists d\ \big(\forall x,y : \mathrm{dist}(x,y) < d\big)\ \mathrm{Eq}(x,y)$$

Ici, $\text{Eq}$ est le prédicat de similarité qu'on a déjà utilisé (mesurant la proximité entre deux points), tandis que $\text{dist}$ est une fonction calculant la distance euclidienne entre deux points.

In [21]:
Eq = ltn.Predicate(func=lambda x, y: torch.exp(-torch.norm(x - y, dim=1))) # predicate measuring similarity

points = torch.rand((50, 2)) # 50 values in [0,1]^2
x = ltn.Variable("x", points)
y = ltn.Variable("y", points)
d = ltn.Variable("d", torch.tensor([.1,.2,.3,.4,.5,.6,.7,.8,.9]))

Pour utiliser une quantification gardée, il suffit de préciser deux arguments supplémentaires au quantificateur : `cond_vars`, la liste des variables LTN impliquées dans le calcul de la condition, et `cond_fn`, la fonction qui calcule le masque booléen à partir de ces variables.

Dans l'exemple ci-dessous, `cond_vars=[x, y, d]` indique que la condition dépend de $x$, $y$ et $d$, tandis que `cond_fn=lambda x, y, d: dist(x, y) < d.value` calcule, pour chaque combinaison, si la distance entre $x$ et $y$ est inférieure au seuil $d$. LTN calcule d'abord $\text{Eq}(x,y)$ normalement sur toutes les combinaisons, calcule en parallèle ce masque booléen, puis ne conserve dans l'agrégation que les valeurs de $\text{Eq}(x,y)$ pour lesquelles le masque vaut vrai. Toutes les autres paires, trop éloignées, sont simplement exclues du calcul.

In [22]:
dist = lambda x, y: torch.unsqueeze(torch.norm(x.value - y.value, dim=1), 1) # function measuring euclidian distance
Exists(d,
      Forall([x, y],
            Eq(x, y),
            cond_vars=[x, y, d],
            cond_fn=lambda x, y, d: dist(x, y) < d.value
            )).value

tensor(0.7599, dtype=torch.float64)

Comme le montre cet exemple, la quantification gardée sert ici à n'agréger que sur les paires de points dont la distance est inférieure à un certain seuil, fixé par la variable $d$. Toutes les autres paires de points, jugées trop éloignées, sont écartées de l'agrégation.

Cette possibilité est particulièrement utile pour propager le gradient (voir le notebook sur l'apprentissage) uniquement sur une partie pertinente du domaine, celle qui vérifie effectivement la condition $m$, plutôt que de diluer l'apprentissage sur des combinaisons qui n'ont pas de sens pour la règle logique concernée.